In [ ]:
"""
|               | LoRA                        | QLoRA                                  |
| ------------- | --------------------------- | -------------------------------------- |
| Base Llama    | Normal precision            | **4-bit**                              |
| Base weights  | Frozen                      | Frozen                                 |
| LoRA adapters | Trainable                   | Trainable                              |
| Quantization  | ❌                          | **✅**                                |
| GPU memory    | Lower than full fine-tuning | **Even lower**                         |
| Main idea     | Train small adapters        | **4-bit model + train small adapters** |

"""
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# 3. QLoRA: load Llama in 4-bit
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True                 # <<------------new change or addition for QLoRA
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    task_type="CAUSAL_LM"
)
dataset = load_dataset(
    "json",
    data_files="train.json"
)
# 7. Training configuration
training_args = SFTConfig(
    output_dir="./llama-qlora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    bf16=True
)
# 8. Trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
    peft_config=lora_config
)
trainer.train()
trainer.save_model("./llama-qlora")
trainer.push_to_hub(
    repo_id="YOUR_USERNAME/llama-3.1-8b-qlora"
)